# §6 — Learned PE vs 1D RoPE in the ViT (with length-extrapolation eval)

Retrains the §3 CLIP-on-EuroSAT setup *from scratch* under two positional-encoding choices for the ViT:
- **`learned`** — the original additive `nn.Parameter` `pos_embed` (length 65 = CLS + 64 patches).
- **`rope`** — 1D RoPE applied to (q, k) inside attention; no additive PE. Patch positions are their 1D index, CLS is position 0.

Both run for the full 20 epochs (`configs/clip_eurosat.yaml`). After training each, the script runs an extra zero-shot eval at **96×96** (12×12 patch grid → 145 tokens vs the 65 seen at training):
- For learned PE, the script bilinearly **interpolates** the 8×8 patch PE to 12×12 inside `ViT.forward` (CLS slot kept separate).
- For RoPE, the cos/sin cache is **extended** to 145 entries (`ViT.set_rope_max_seq_len`).

**Compute:** 2× the standard §3 run. On L4, plan ~20–30 minutes per run.

## 1. Install dependencies

In [1]:
%%capture
!pip -q install -U transformers datasets "sentence-transformers<4.0" accelerate tqdm matplotlib wandb pyyaml einops
!pip -q install --force-reinstall --no-deps --no-cache-dir pillow==11.3.0

## 2. Mount Drive and set up the repo path

In [2]:
import os, sys
from pathlib import Path

USE_DRIVE = True
DRIVE_REPO_ROOT = Path('/content/drive/MyDrive/caltech/junior/hw3/')  # edit if needed

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    REPO_ROOT = DRIVE_REPO_ROOT
else:
    REPO_ROOT = Path('/content/hw3')

assert REPO_ROOT.exists(), f'Repo root does not exist: {REPO_ROOT}'
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

Mounted at /content/drive
cwd = /content/drive/MyDrive/caltech/junior/hw3


In [3]:
import torch, torchvision

print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
torchvision.ops.nms(torch.zeros(0, 4), torch.zeros(0), 0.5)

torch: 2.10.0+cu128 | cuda: True


tensor([], dtype=torch.int64)

## 3. (Optional) Log to W&B

Same as the §3 notebook. Skip if you don't care about W&B for this comparison.

In [7]:
USE_WANDB = False
if USE_WANDB:
    import wandb
    wandb.login()

## 4. Configure the sweep

In [4]:
import yaml

CONFIG_PATH = REPO_ROOT / 'configs' / 'clip_eurosat.yaml'
assert CONFIG_PATH.exists()
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

TRAIN_IMG_SIZE = cfg['vit']['img_size']     # 64
EVAL_IMG_SIZE = 96                           # 12x12 patch grid -> 145 tokens
PATCH_SIZE = cfg['vit']['patch_size']        # 8
POS_ENCODINGS = ['learned', 'rope']

print(f'train img_size  = {TRAIN_IMG_SIZE}  ({TRAIN_IMG_SIZE // PATCH_SIZE}x{TRAIN_IMG_SIZE // PATCH_SIZE} grid)')
print(f'eval img_size   = {EVAL_IMG_SIZE}   ({EVAL_IMG_SIZE  // PATCH_SIZE}x{EVAL_IMG_SIZE  // PATCH_SIZE} grid)')
print(f'PE methods      = {POS_ENCODINGS}')

train img_size  = 64  (8x8 grid)
eval img_size   = 96   (12x12 grid)
PE methods      = ['learned', 'rope']


## 5. Train both PE variants and run the extrapolation eval

Each run writes the §3 outputs (`best.pt`, `history.pt`) to a per-PE directory in `/content/runs/clip_eurosat_<pe>/`, then single-shot-syncs to Drive. Resumable: a prior `history.pt` on Drive is reused unless `OVERWRITE = True`.

In [8]:
import argparse, shutil, time

from scripts.pretrain_clip import train

OVERWRITE = False
device = 'cuda' if torch.cuda.is_available() else 'cpu'

pe_results = {}
for pe in POS_ENCODINGS:
    local_dir = Path(f'/content/runs/clip_eurosat_{pe}')
    drive_dir = REPO_ROOT / 'runs' / local_dir.name
    history_path = drive_dir / 'history.pt'

    if not OVERWRITE and history_path.exists():
        hist = torch.load(history_path, map_location='cpu', weights_only=False)
        pe_results[pe] = hist
        print(f'[{pe}] reusing cached history  '
              f'(test_acc={hist["test_acc"]:.4f}, '
              f'extra={hist.get("extrapolation", {}).get("test_acc", None)})')
        continue

    local_dir.mkdir(parents=True, exist_ok=True)
    args = argparse.Namespace(
        config=CONFIG_PATH,
        output_dir=local_dir,
        device=device,
        wandb=USE_WANDB,
        wandb_project='cs148b-hw3-rope-vs-learned',
        wandb_run_name=f'eurosat-{pe}',
        pos_encoding=pe,
        eval_img_size=EVAL_IMG_SIZE,
    )
    print(f'\n========== pos_encoding={pe} ==========')
    t0 = time.time()
    history = train(cfg, args)
    print(f'[{pe}] wall = {(time.time() - t0)/60:.1f} min  '
          f'test_acc={history["test_acc"]:.4f}  '
          f'extra@{EVAL_IMG_SIZE}={history["extrapolation"]["test_acc"]:.4f}')
    pe_results[pe] = history

    drive_dir.mkdir(parents=True, exist_ok=True)
    for f in local_dir.glob('*'):
        shutil.copy2(f, drive_dir / f.name)
    print(f'synced {local_dir} -> {drive_dir}')


========== pos_encoding=learned ==========


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

...

epoch 19/20:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7cc5343fc860>
Traceback (most recent call last):
Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7cc5343fc860>
    self._shutdown_workers()Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    Exception ignored in: self._shutdown_workers()    <function _MultiProcessingDataLoaderIter.__del__ at 0x7cc5343fc860>if w.is_alive():


  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Traceback (most recent call last):
Exception ignored in:        File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1

[epoch 19/20] train_loss=3.3485  val_acc=0.8967  lr=2.88e-06  (35.5s)


epoch 20/20:   0%|          | 0/50 [00:00<?, ?it/s]

[epoch 20/20] train_loss=3.3408  val_acc=0.9004  lr=0.00e+00  (15.8s)
[final] best_val_acc=0.9041  test_acc=0.9060
[extrapolation] running zero-shot eval at img_size=96
[extrapolation] img_size=96  val_acc=0.7649  test_acc=0.7764
[rope] wall = 10.1 min  test_acc=0.9060  extra@96=0.7764
synced /content/runs/clip_eurosat_rope -> /content/drive/MyDrive/caltech/junior/hw3/runs/clip_eurosat_rope


## 6. 2-row deliverable table

cols = [
    ('pos_encoding',       12),
    (f'val@{TRAIN_IMG_SIZE}',  10),
    (f'test@{TRAIN_IMG_SIZE}', 11),
    (f'val@{EVAL_IMG_SIZE}',   10),
    (f'test@{EVAL_IMG_SIZE}',  11),
    ('delta_test',          11),
]
header = ' | '.join(f'{n:>{w}}' for n, w in cols)
print()
print(header)
print('-' * len(header))
for pe in POS_ENCODINGS:
    h = pe_results.get(pe)
    if h is None:
        continue
    train_val = h['best_val_acc']
    train_test = h['test_acc']
    extra = h.get('extrapolation', {})
    eval_val = extra.get('val_acc', float('nan'))
    eval_test = extra.get('test_acc', float('nan'))
    delta = eval_test - train_test
    row = [
        f'{pe:>12}',
        f'{train_val:>10.4f}',
        f'{train_test:>11.4f}',
        f'{eval_val:>10.4f}',
        f'{eval_test:>11.4f}',
        f'{delta:>+11.4f}',
    ]
    print(' | '.join(row))

## 7. Writeup notes (3–4 sentences)

Fill in using your actual numbers. Things to cover:

- **At the training size (64×64), both PEs land in a similar accuracy range** — neither dominates on the *in-distribution* task, because both encode enough position to let attention bind patches to a grid layout. (Cite the gap, if any.)

- **At 96×96 (≈2.25× more tokens), RoPE extrapolates much better than learned-PE-with-interpolation.** RoPE only ever sees relative offsets between patches via the (q, k) rotation, and those offsets are simply 1D index differences — the math is the same whether `T=65` or `T=145`. Learned PE, by contrast, has to *invent* PE values for positions the network never saw, and bilinear interpolation on the 8×8 PE grid is at best a smooth guess: it preserves coarse spatial structure but produces PE vectors with different distributional statistics than the trained ones, which downstream LayerNorms / attention heads weren't tuned for.

- **The accuracy drop columns make the takeaway concrete.** Expect learned-PE's `Δtest = test@96 − test@64` to be a substantial negative number; RoPE's `Δtest` should be much smaller (often near zero, sometimes slightly positive — more tokens = more information per image, and RoPE's relative formulation makes that extra information actually usable).

- **What this tells you about the choice of PE.** When you need length generalization at inference (different image resolution, longer sequences, etc.), relative / rotary PEs win on principle — there's no PE table to outgrow. The price is a small per-attention-layer cost (apply RoPE to q and k) and that you have to think about the 2D layout explicitly when you go from 1D-index RoPE to a real grid-aware encoding (the next problem).

Optional: note that the 8×8 → 12×12 PE interpolation is the standard way to *deploy* a learned-PE ViT at a new resolution (this is what timm does), so the comparison is fair — RoPE's robustness comes from the encoding itself, not from a weaker baseline.